In [3]:

# 1) Naive space-based tokens
# 2) Manually corrected tokens (punctuation + clitics)
# 3) Tool tokens (NLTK)
# 4) Differences between manual vs tool


import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize

# --- download once (quiet) ---
nltk.download("punkt", quiet=True)

# ----------------------------
# Paragraph (3 sentences)
# ----------------------------
text = (
    "Chicago’s winter is crazy—it’s -10°C today! "
    "I can’t believe my friend’s car won’t start; he’s calling AAA. "
    "Anyway, I’m heading to New York City tomorrow, in a hurry."
)

# --- FIX 1: normalize smart quotes to normal apostrophe ---
text = text.replace("’", "'").replace("‘", "'")

print("PARAGRAPH:\n", text, "\n")

# ----------------------------
# 1) Naïve space-based tokenization
# ----------------------------
naive_tokens = text.split()
print("1) Naïve space-based tokens:")
print(naive_tokens, "\n")

# ----------------------------
# 2) Manual tokenization (clean + simple)
#    - split punctuation as separate tokens
#    - split common clitics: 's, 'm, 're, 've, 'd, 'll, n't
# ----------------------------
def manual_tokenize(t: str):
    # put spaces around punctuation (keep dash/em-dash as its own token)
    t = re.sub(r"([.!?,;:()\"—])", r" \1 ", t)

    # split contractions/clitics: he's -> he 's, I'm -> I 'm, they've -> they 've
    t = re.sub(r"\b(\w+)'(s|m|re|ve|d|ll)\b", r"\1 ' \2", t)

    # split n't: can't -> ca n't, won't -> wo n't (simple English rule)
    t = re.sub(r"\b(\w+)n't\b", r"\1 n't", t)

    # clean spaces
    t = re.sub(r"\s+", " ", t).strip()
    return t.split()

manual_tokens = manual_tokenize(text)
print("2) Manually corrected tokens:")
print(manual_tokens, "\n")

# ----------------------------
# 3) Tool tokenization (NLTK)
# ----------------------------
tool_tokens = word_tokenize(text)
print("3) NLP tool tokens (NLTK):")
print(tool_tokens, "\n")

# ----------------------------
# Compare manual vs tool (order-based + set-based)
# ----------------------------
print("=== Differences (manual vs tool) ===")

# Set differences (quick view)
only_manual = sorted(set(manual_tokens) - set(tool_tokens))
only_tool = sorted(set(tool_tokens) - set(manual_tokens))

print("\nTokens only in MANUAL (not in tool):")
print(only_manual)

print("\nTokens only in TOOL (not in manual):")
print(only_tool)

# Count differences (more accurate than set if duplicates exist)
manual_counts = Counter(manual_tokens)
tool_counts = Counter(tool_tokens)

count_diffs = []
for tok in sorted(set(manual_tokens) | set(tool_tokens)):
    if manual_counts[tok] != tool_counts[tok]:
        count_diffs.append((tok, manual_counts[tok], tool_counts[tok]))

print("\nToken count differences (token, manual_count, tool_count):")
print(count_diffs if count_diffs else "No count differences")

print("\nNOTES:")
print("- Naive tokenization keeps punctuation attached (e.g., 'today!').")
print("- Manual tokenization splits punctuation and common clitics (e.g., \"I'm\" -> I ' m, \"can't\" -> can n't).")
print("- NLTK uses its own built-in rules; it may split some forms slightly differently.")
print("- MWEs like 'New York City' stay as separate tokens unless you add MWE/NER logic.")


PARAGRAPH:
 Chicago's winter is crazy—it's -10°C today! I can't believe my friend's car won't start; he's calling AAA. Anyway, I'm heading to New York City tomorrow, in a hurry. 

1) Naïve space-based tokens:
["Chicago's", 'winter', 'is', "crazy—it's", '-10°C', 'today!', 'I', "can't", 'believe', 'my', "friend's", 'car', "won't", 'start;', "he's", 'calling', 'AAA.', 'Anyway,', "I'm", 'heading', 'to', 'New', 'York', 'City', 'tomorrow,', 'in', 'a', 'hurry.'] 

2) Manually corrected tokens:
['Chicago', "'", 's', 'winter', 'is', 'crazy', '—', 'it', "'", 's', '-10°C', 'today', '!', 'I', 'ca', "n't", 'believe', 'my', 'friend', "'", 's', 'car', 'wo', "n't", 'start', ';', 'he', "'", 's', 'calling', 'AAA', '.', 'Anyway', ',', 'I', "'", 'm', 'heading', 'to', 'New', 'York', 'City', 'tomorrow', ',', 'in', 'a', 'hurry', '.'] 

3) NLP tool tokens (NLTK):
['Chicago', "'s", 'winter', 'is', 'crazy—it', "'s", '-10°C', 'today', '!', 'I', 'ca', "n't", 'believe', 'my', 'friend', "'s", 'car', 'wo', "n't", 's